In [1]:
import mlflow
from mlflow.models import infer_signature


In [ ]:
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Load the Iris dataset
X, y = datasets.load_iris(return_X_y=True)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,
}

# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

# Predict on the test set
y_pred = lr.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)


Initiate an MLflow run context to start a new run that we will log the model and metadata to.

Log model parameters and performance metrics.

Tag the run for easy retrieval.

Register the model in the MLflow Model Registry while logging (saving) the model.


In [ ]:
mlflow.set_tracking_uri(uri = "http://127.0.0.1:5000")
mlflow.set_experiment("MLflow Quickstart")

In [ ]:
with mlflow.start_run():
    mlflow.log_params(params)

    mlflow.log_metric("accuracy", accuracy)

    mlflow.set_tag("Training Info", "Basic LR model for Iris dataset")

    signature = infer_signature(X_train, lr.predict(X_train))

    model_info = mlflow.sklearn.log_model(
        sk_model = lr,
        artifact_path = "iris_model",
        signature = signature,
        input_example = X_train,
        registered_model_name = "tracking_quickstart"
    )
    

### Load the model as a Python Function (pyfunc) and use it for inference

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

In [12]:
predictions = loaded_model.predict(X_test)

In [13]:
iris_feature_names = datasets.load_iris().feature_names

In [ ]:
result_df = pd.DataFrame(X_test, columns=iris_feature_names)
result_df["actual_class"] = y_test
result_df["predicted_class"] = predictions

result_df.head()